# Model selection criteria

_Machine Learning — more_

**Reward fit, punish complexity. The best model balances the two.**

A bigger model always fits the training data better. But it may just memorize noise.

     So we need a score that rewards good fit but punishes extra parameters.

---

This notebook is a practice scaffold for the **Model selection criteria** lesson. We build the idea up one step at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make data whose true shape we know

We generate 60 points from a known quadratic (`2 + 1.5x - 0.8x^2`) and add a little Gaussian noise. Because *we* picked the formula, we know the honest answer is **degree 2** — anything fancier is just chasing the noise. This is our controlled test bed for the selection criteria.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(0)
x = np.linspace(-3, 3, 60)

# True relationship is a degree-2 curve, plus a little Gaussian noise.
y = 2 + 1.5*x - 0.8*x**2 + 0.5*rng.randn(60)
n = len(y)

print("made", n, "points from a true degree-2 curve")

### Step 2 — Score each candidate degree with AIC and BIC

For each polynomial degree from 1 to 7 we fit a linear regression and measure the residual sum of squares (RSS) — pure goodness of fit, which only ever improves with more parameters. **AIC** and **BIC** convert RSS into a log-likelihood and then add a penalty for the number of parameters `k`, so a model only wins if its better fit outweighs its extra complexity. BIC penalizes complexity harder than AIC because its penalty grows with `log(n)`.

In [ ]:
print("degree   RSS      AIC      BIC")
best_deg, best_aic = None, np.inf

for deg in range(1, 8):
    Xp = PolynomialFeatures(deg).fit_transform(x.reshape(-1, 1))
    yhat = LinearRegression().fit(Xp, y).predict(Xp)
    rss = float(np.sum((y - yhat)**2))

    k = Xp.shape[1] + 1                          # number of coefs + the noise variance
    lnL = -0.5*n*(np.log(2*np.pi*rss/n) + 1)     # Gaussian log-likelihood
    aic = 2*k - 2*lnL                            # AIC: penalty 2 per parameter
    bic = k*np.log(n) - 2*lnL                    # BIC: penalty log(n) per parameter

    print("%5d  %7.3f  %7.2f  %7.2f" % (deg, rss, aic, bic))
    if aic < best_aic:
        best_aic, best_deg = aic, deg

### Step 3 — Read off the winner

The degree with the lowest AIC is the one the criterion prefers. Notice that RSS keeps shrinking as the degree climbs, yet AIC bottoms out near the *true* degree 2 — the complexity penalty is what stops us from rewarding overfitting.

In [ ]:
print("best degree by AIC = %d" % best_deg)

## Visualize the data & results

_Which model complexity is best on real data?_

### Step 1 — Load real data and sweep model complexity

Now we leave the toy curve and use 569 real breast-cancer patients with 30 standardized features. We use k-nearest-neighbors, where the number of neighbors `k` is a complexity dial: a **small** `k` is a very flexible, complex model, while a **large** `k` is smooth and simple. For each `k` we record the 5-fold cross-validated accuracy — held-out performance, the honest judge of complexity.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# REAL data: 569 breast-cancer patients, 30 features.
bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)

ks, cvs = [], []
for k in [1, 3, 5, 7, 9, 15, 25, 45]:
    model = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(model, X, bc.target, cv=5).mean()
    ks.append(k)                 # small k = more complex model
    cvs.append(score)

### Step 2 — Plot accuracy against complexity

Plotting cross-validated accuracy versus `k` traces the classic complexity curve. Too-complex (tiny `k`) fits noise and generalizes poorly; too-simple (huge `k`) underfits. The **peak** in the middle is the sweet spot — the same fit-versus-complexity trade-off that AIC/BIC encoded, now read straight off held-out data.

In [ ]:
plt.plot(ks, cvs, 'o-', color='#4ea1ff', label='CV accuracy')
plt.xlabel('neighbors k (small k = more complex model)')
plt.ylabel('5-fold cross-validated accuracy')
plt.title('Breast cancer: CV accuracy vs k-NN complexity (peak = best)')
plt.legend()
plt.show()